<a href="https://colab.research.google.com/github/korkutanapa/DCASE2025TASK2/blob/main/ORJ_DCASE_FEATURE_SELECTION_LAST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
"""
JOINT k + MACHINE-SPECIFIC TDA FEATURE ORACLE SEARCH
=====================================================

Purpose
-------
For each labeled DCASE development machine, jointly search:
    * k in {3, 5, 10, 20, 30}
    * TDA feature subset

The detector is a normal-reference kNN detector:
    anomaly_score(x) = mean distance to the k nearest NORMAL train samples

IMPORTANT SCORE DIRECTION
-------------------------
Larger distance = more anomalous.
The code NEVER flips the score and NEVER replaces AUC with max(AUC, 1-AUC).
If a subset gives AUC < 0.5, it is genuinely treated as a poor subset.

Study type
----------
This is a SUPERVISED DEVELOPMENT / ORACLE capability study.
Development test labels are intentionally used to choose both feature subset
and k. Therefore the resulting performance is an optimistic development upper
bound and is NOT a deployable unseen-machine feature-selection rule.

Search
------
* 1 feature : exhaustive over all usable TDA features, for all k values
* 2 features: exhaustive over all pairs, for all k values
* 3+        : k-preserving multi-beam forward search
               - one subset evaluation finds the 30 nearest neighbors once
               - scores for k=3,5,10,20,30 are derived from the same distances
               - top candidates are retained separately for EACH k, then unioned
             This prevents the beam from becoming biased toward one k value.
* Optional final same-size swap refinement for the best subset of each k.

Default objective
-----------------
DCASE-oriented machine score:
    HM(AUC_source, AUC_target, pAUC@0.1)
Tie-breakers:
    pAUC@0.1
    min(AUC_source, AUC_target)
    AUC_all
    fewer features
    smaller k

Set OBJECTIVE = "auc_all" below if you want the original overall-AUC-first
oracle instead.

Outputs
-------
/content/joint_k_machine_specific_tda_oracle/
    joint_k_machine_specific_tda_oracle.xlsx
    best_oracle_feature_pool.json

Excel sheets:
    Best_Overall
    Best_By_Machine_K
    Best_By_Size_K
    Top_Singles
    Top_Pairs
    Beam_History
    Expansion_Pool
    Feature_Stability
    Config

The JSON contains the union of features selected by the best machine x k
solutions, plus the selected subset for every machine and k.
"""

from __future__ import annotations

from pathlib import Path
from itertools import combinations, count
from collections import Counter
import heapq
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")


# =============================================================================
# CONFIGURATION
# =============================================================================

DATA_DIR = Path("/content")

K_VALUES = (3, 5, 10, 20, 30)
MAX_K = max(K_VALUES)
PAUC_MAX_FPR = 0.10

# "dcase_hm" = HM(AUC_source, AUC_target, pAUC@0.1)
# "auc_all"  = overall AUC first, then pAUC
OBJECTIVE = "dcase_hm"

# Search subset sizes 1..MAX_FEATURES.
MAX_FEATURES = 20

# Higher-order search settings (sizes >= 3).
# Top BEAM_PER_K subsets are retained separately for each k, then unioned.
BEAM_PER_K = 10
SAVE_TOP_PER_SIZE_PER_K = 10

# Expansion pool is derived from strong exhaustive singles/pairs across ALL k.
TOP_SINGLE_PER_K_FOR_POOL = 50
TOP_PAIR_PER_K_FOR_POOL = 200
MAX_EXPANSION_POOL = 120

# If True, all usable TDA features are allowed in 3+ feature expansion.
# This is stronger but can become much slower.
EXPAND_WITH_ALL_VALID_FEATURES = False

# Mild train-normal redundancy control for 3+ subsets only.
USE_REDUNDANCY_FILTER = True
MAX_ABS_TRAIN_CORR = 0.995

# Final local refinement of the best subset for each machine x k.
ENABLE_FINAL_SWAP_REFINEMENT = True
SWAP_CANDIDATE_LIMIT = 60
MAX_SWAP_PASSES = 1

# Number of top exhaustive pair rows retained per machine x k.
# The full pair space is evaluated, but only strongest rows are kept in memory.
PAIR_KEEP_TOP_PER_K = max(500, TOP_PAIR_PER_K_FOR_POOL, BEAM_PER_K)

# cKDTree query workers. -1 uses all available cores in supported scipy versions.
KD_WORKERS = -1

# Console progress interval for exhaustive pairs.
PAIR_PROGRESS_EVERY = 5000

OUTPUT_DIR = Path("/content/joint_k_machine_specific_tda_oracle")
OUTPUT_XLSX = OUTPUT_DIR / "joint_k_machine_specific_tda_oracle.xlsx"
OUTPUT_JSON = OUTPUT_DIR / "best_oracle_feature_pool.json"

# TDA columns intentionally excluded, matching the original notebook.
EXCLUDED_TDA_FEATURES = {
    "H0_points_raw",
    "H1_points_raw",
}


# =============================================================================
# DATASET DETECTION
# =============================================================================

FILE_PATTERN = re.compile(
    r"^cubical_mel_tda_features_dev_(?P<machine>.+?)_"
    r"(?P<split>train|test)(?:\(\d+\))?$",
    flags=re.IGNORECASE,
)


def detect_datasets(data_dir: Path) -> dict:
    """Find newest train/test Excel pair for every development machine."""

    records = []

    for path in data_dir.rglob("*.xlsx"):
        match = FILE_PATTERN.match(path.stem)
        if not match:
            continue

        records.append(
            {
                "machine_raw": match.group("machine"),
                "machine_key": match.group("machine").lower(),
                "split": match.group("split").lower(),
                "path": path,
                "mtime": path.stat().st_mtime,
            }
        )

    if not records:
        raise FileNotFoundError(
            f"No DCASE development train/test Excel files were found under {data_dir}."
        )

    # Keep newest duplicate for each machine/split.
    selected = {}
    for rec in records:
        key = (rec["machine_key"], rec["split"])
        old = selected.get(key)
        if old is None or rec["mtime"] > old["mtime"]:
            selected[key] = rec

    datasets = {}
    machine_keys = sorted({k[0] for k in selected})

    for machine_key in machine_keys:
        train_rec = selected.get((machine_key, "train"))
        test_rec = selected.get((machine_key, "test"))

        if train_rec is None or test_rec is None:
            print(
                f"WARNING: incomplete pair ignored for {machine_key}: "
                f"train={train_rec is not None}, test={test_rec is not None}"
            )
            continue

        display_name = train_rec["machine_raw"]
        datasets[machine_key] = {
            "display_name": display_name,
            "train": train_rec["path"],
            "test": test_rec["path"],
        }

    if not datasets:
        raise FileNotFoundError(
            "No complete train/test machine pairs were found. "
            "Both train and test feature Excel files are required for kNN."
        )

    return datasets


# =============================================================================
# LABEL AND DOMAIN HELPERS
# =============================================================================


def labels_to_binary(series: pd.Series) -> np.ndarray:
    """Convert normal/anomaly labels to 0/1, where anomaly=1."""

    if pd.api.types.is_numeric_dtype(series):
        values = pd.to_numeric(series, errors="coerce")
        if values.isna().any():
            raise ValueError("Some numeric labels could not be interpreted.")
        return (values > 0).astype(np.int8).to_numpy()

    text = series.astype(str).str.strip().str.lower()
    output = []
    unknown = set()

    anomaly_words = {
        "1", "anomaly", "anomalous", "abnormal", "fault", "faulty", "ng"
    }
    normal_words = {"0", "normal", "healthy", "ok"}

    for value in text:
        if (
            value in anomaly_words
            or "anomal" in value
            or "abnormal" in value
            or "fault" in value
        ):
            output.append(1)
        elif value in normal_words or "normal" in value or "healthy" in value:
            output.append(0)
        else:
            output.append(-1)
            unknown.add(value)

    if unknown:
        raise ValueError(f"Unknown labels: {sorted(unknown)}")

    return np.asarray(output, dtype=np.int8)


def _normalize_domain_value(value) -> str:
    text = str(value).strip().lower()
    if "source" in text:
        return "source"
    if "target" in text:
        return "target"
    return "unknown"


def infer_test_domains(test_df: pd.DataFrame) -> np.ndarray:
    """
    Infer source/target domain.

    Priority:
      1) explicit domain-like column
      2) file_id / file_path / filename / path text

    The uploaded DCASE files encode domain in file_id/file_path, e.g.
      section_00_source_test_anomaly_....wav
    """

    explicit_candidates = [
        "domain",
        "domain_label",
        "source_target",
        "data_domain",
        "dataset_domain",
    ]

    lower_to_real = {str(c).lower(): c for c in test_df.columns}

    for candidate in explicit_candidates:
        if candidate in lower_to_real:
            col = lower_to_real[candidate]
            domains = test_df[col].map(_normalize_domain_value).to_numpy()
            known_rate = np.mean(domains != "unknown")
            if known_rate >= 0.95:
                return domains

    text_columns = []
    for candidate in ["file_id", "file_path", "filename", "path", "wav_path"]:
        if candidate in lower_to_real:
            text_columns.append(lower_to_real[candidate])

    if not text_columns:
        return np.asarray(["unknown"] * len(test_df), dtype=object)

    combined = test_df[text_columns[0]].astype(str)
    for col in text_columns[1:]:
        combined = combined + " " + test_df[col].astype(str)

    return combined.map(_normalize_domain_value).to_numpy()


# =============================================================================
# METRICS AND RANKING
# =============================================================================


def safe_auc(y: np.ndarray, scores: np.ndarray, max_fpr=None) -> float:
    if len(y) == 0 or len(np.unique(y)) < 2:
        return float("nan")
    return float(roc_auc_score(y, scores, max_fpr=max_fpr))


def harmonic_mean(values) -> float:
    values = np.asarray(values, dtype=float)
    if len(values) == 0 or np.any(~np.isfinite(values)) or np.any(values <= 0):
        return float("nan")
    return float(len(values) / np.sum(1.0 / values))


def compute_metrics(
    y_test: np.ndarray,
    domains: np.ndarray,
    scores: np.ndarray,
) -> dict:
    """Compute overall and DCASE-oriented development metrics."""

    auc_all = safe_auc(y_test, scores)
    pauc_01 = safe_auc(y_test, scores, max_fpr=PAUC_MAX_FPR)

    source_mask = domains == "source"
    target_mask = domains == "target"

    auc_source = safe_auc(y_test[source_mask], scores[source_mask])
    auc_target = safe_auc(y_test[target_mask], scores[target_mask])

    dcase_hm = harmonic_mean([auc_source, auc_target, pauc_01])

    if np.isfinite(auc_source) and np.isfinite(auc_target):
        min_domain_auc = float(min(auc_source, auc_target))
    else:
        min_domain_auc = float("nan")

    return {
        "auc_all": auc_all,
        "auc_source": auc_source,
        "auc_target": auc_target,
        "pauc_01": pauc_01,
        "dcase_hm": dcase_hm,
        "min_domain_auc": min_domain_auc,
    }


def _finite_or_neg_inf(value) -> float:
    try:
        value = float(value)
    except Exception:
        return -math.inf
    return value if np.isfinite(value) else -math.inf


def ranking_tuple(row: dict) -> tuple:
    """
    Higher tuple = better.

    DCASE mode follows:
      HM(source AUC, target AUC, pAUC)
      -> pAUC
      -> min(source AUC, target AUC)
      -> AUC(all)
      -> fewer features
      -> smaller k
    """

    if OBJECTIVE == "dcase_hm":
        return (
            _finite_or_neg_inf(row["dcase_hm"]),
            _finite_or_neg_inf(row["pauc_01"]),
            _finite_or_neg_inf(row["min_domain_auc"]),
            _finite_or_neg_inf(row["auc_all"]),
            -int(row["n_features"]),
            -int(row["k"]),
        )

    if OBJECTIVE == "auc_all":
        return (
            _finite_or_neg_inf(row["auc_all"]),
            _finite_or_neg_inf(row["pauc_01"]),
            _finite_or_neg_inf(row["dcase_hm"]),
            -int(row["n_features"]),
            -int(row["k"]),
        )

    raise ValueError(f"Unknown OBJECTIVE={OBJECTIVE!r}")


def canonical_subset(features) -> tuple:
    """Feature order is irrelevant for Euclidean kNN."""
    return tuple(sorted(set(features)))


def subset_to_text(subset) -> str:
    return " | ".join(subset)


# =============================================================================
# TRAIN-ONLY PREPROCESSING
# =============================================================================


def prepare_machine_data(train_df: pd.DataFrame, test_df: pd.DataFrame) -> dict:
    """
    Prepare one machine.

    Preprocessing is fitted ONLY on normal train data:
      * median imputation
      * remove constant / near-constant features
      * StandardScaler

    All available normal training samples are used as the kNN reference bank.
    Test labels are used only for oracle evaluation/selection.
    """

    if "label" not in test_df.columns:
        raise ValueError("Test file does not contain a 'label' column.")

    # If a train label exists, explicitly keep only normal rows.
    # DCASE train files are expected to be normal-only already.
    if "label" in train_df.columns:
        try:
            y_train = labels_to_binary(train_df["label"])
            if np.any(y_train == 1):
                print(
                    f"  WARNING: removing {int(np.sum(y_train == 1))} "
                    "non-normal rows from train reference bank."
                )
                train_df = train_df.loc[y_train == 0].reset_index(drop=True)
        except Exception:
            # Do not fail merely because train labels use an unexpected convention.
            pass

    features = [
        col
        for col in train_df.columns
        if (
            col in test_df.columns
            and str(col).startswith(("H0_", "H1_"))
            and col not in EXCLUDED_TDA_FEATURES
        )
    ]

    if len(features) < 2:
        raise ValueError("Fewer than two common TDA feature columns were found.")

    train_num = (
        train_df[features]
        .replace([np.inf, -np.inf], np.nan)
        .apply(pd.to_numeric, errors="coerce")
    )
    test_num = (
        test_df[features]
        .replace([np.inf, -np.inf], np.nan)
        .apply(pd.to_numeric, errors="coerce")
    )

    nonempty = [f for f in features if not train_num[f].isna().all()]
    train_num = train_num[nonempty]
    test_num = test_num[nonempty]

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(train_num)
    X_test = imputer.transform(test_num)

    std = np.std(X_train, axis=0)
    valid_mask = np.isfinite(std) & (std > 1e-12)

    feature_names = [
        feature for feature, keep in zip(nonempty, valid_mask) if keep
    ]

    X_train = X_train[:, valid_mask]
    X_test = X_test[:, valid_mask]

    if len(feature_names) < 2:
        raise ValueError("Fewer than two nonconstant TDA features remain.")

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    y_test = labels_to_binary(test_df["label"])
    if len(np.unique(y_test)) < 2:
        raise ValueError("Test labels must contain normal and anomaly samples.")

    domains = infer_test_domains(test_df)

    if OBJECTIVE == "dcase_hm":
        for domain_name in ("source", "target"):
            mask = domains == domain_name
            if np.sum(mask) == 0 or len(np.unique(y_test[mask])) < 2:
                raise ValueError(
                    f"OBJECTIVE='dcase_hm' requires both normal and anomaly "
                    f"test samples in domain={domain_name!r}. "
                    "Domain is inferred from file_id/file_path or an explicit domain column."
                )

    if len(X_train) < MAX_K:
        raise ValueError(
            f"At least {MAX_K} normal train samples are required; found {len(X_train)}."
        )

    feature_to_index = {f: i for i, f in enumerate(feature_names)}

    # Correlation is computed only on normal train data.
    corr = np.corrcoef(X_train, rowvar=False).astype(np.float32)
    corr[~np.isfinite(corr)] = 0.0

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_test": y_test,
        "domains": domains,
        "feature_names": feature_names,
        "feature_to_index": feature_to_index,
        "corr": corr,
    }


# =============================================================================
# SUBSET EVALUATION
# =============================================================================


def query_knn_distances(tree: cKDTree, X_test: np.ndarray, k: int) -> np.ndarray:
    """Compatibility wrapper for scipy versions with/without workers=."""
    try:
        distances, _ = tree.query(X_test, k=k, workers=KD_WORKERS)
    except TypeError:
        distances, _ = tree.query(X_test, k=k)

    distances = np.asarray(distances, dtype=np.float64)
    if distances.ndim == 1:
        distances = distances[:, None]
    return distances


def evaluate_subset_all_k(
    machine_name: str,
    subset,
    data: dict,
) -> list[dict]:
    """
    Evaluate one feature subset for ALL requested k values.

    Efficiency trick:
      query MAX_K=30 neighbors once,
      then cumulative means give scores for k=3,5,10,20,30.
    """

    subset = canonical_subset(subset)
    index_map = data["feature_to_index"]

    if not all(feature in index_map for feature in subset):
        return []

    indices = [index_map[feature] for feature in subset]
    X_train = data["X_train"][:, indices]
    X_test = data["X_test"][:, indices]

    tree = cKDTree(X_train)
    distances = query_knn_distances(tree, X_test, MAX_K)
    cumulative = np.cumsum(distances, axis=1)

    rows = []
    for k in K_VALUES:
        # IMPORTANT: larger mean distance = larger anomaly score.
        # No sign reversal and no AUC flipping is performed.
        scores = cumulative[:, k - 1] / float(k)

        metrics = compute_metrics(
            data["y_test"],
            data["domains"],
            scores,
        )

        rows.append(
            {
                "machine": machine_name,
                "k": int(k),
                "n_features": len(subset),
                "subset": subset,
                "features": subset_to_text(subset),
                **metrics,
            }
        )

    return rows


# =============================================================================
# REDUNDANCY FILTER
# =============================================================================


def subset_is_redundant(subset, data: dict) -> bool:
    if not USE_REDUNDANCY_FILTER or len(subset) < 2:
        return False

    idx = [data["feature_to_index"][f] for f in subset]
    subcorr = np.abs(data["corr"][np.ix_(idx, idx)])

    upper = subcorr[np.triu_indices(len(idx), k=1)]
    return bool(np.any(upper >= MAX_ABS_TRAIN_CORR))


# =============================================================================
# TOP-N HEAP HELPERS
# =============================================================================


_heap_counter = count()


def push_top(heap: list, row: dict, limit: int):
    """Keep only the strongest `limit` rows according to ranking_tuple."""
    item = (ranking_tuple(row), next(_heap_counter), row)

    if len(heap) < limit:
        heapq.heappush(heap, item)
    elif item[0] > heap[0][0]:
        heapq.heapreplace(heap, item)


def sorted_heap_rows(heap: list) -> list[dict]:
    return sorted(
        [item[2] for item in heap],
        key=ranking_tuple,
        reverse=True,
    )


def best_row(rows: list[dict]) -> dict:
    if not rows:
        raise ValueError("No rows available for best-row selection.")
    return max(rows, key=ranking_tuple)


# =============================================================================
# EXHAUSTIVE SINGLE SEARCH
# =============================================================================


def exhaustive_single_search(machine_name: str, data: dict) -> list[dict]:
    print("  Exhaustive single-feature search ...")

    rows = []
    features = data["feature_names"]

    for i, feature in enumerate(features, start=1):
        rows.extend(evaluate_subset_all_k(machine_name, (feature,), data))

        if i % 25 == 0 or i == len(features):
            print(f"    singles {i}/{len(features)}")

    return rows


# =============================================================================
# EXHAUSTIVE PAIR SEARCH
# =============================================================================


def exhaustive_pair_search(machine_name: str, data: dict) -> dict[int, list[dict]]:
    """
    Evaluate every two-feature combination.

    All pairs are genuinely evaluated, but only the strongest
    PAIR_KEEP_TOP_PER_K rows for each k are retained in memory/output.
    """

    features = data["feature_names"]
    n_pairs = len(features) * (len(features) - 1) // 2

    print(f"  Exhaustive pair search: {n_pairs:,} pairs ...")

    heaps = {k: [] for k in K_VALUES}

    for pair_no, pair in enumerate(combinations(features, 2), start=1):
        rows = evaluate_subset_all_k(machine_name, pair, data)

        for row in rows:
            push_top(heaps[row["k"]], row, PAIR_KEEP_TOP_PER_K)

        if pair_no % PAIR_PROGRESS_EVERY == 0 or pair_no == n_pairs:
            print(f"    pairs {pair_no:,}/{n_pairs:,}")

    return {k: sorted_heap_rows(heaps[k]) for k in K_VALUES}


# =============================================================================
# EXPANSION POOL
# =============================================================================


def build_expansion_pool(
    single_rows: list[dict],
    pair_top_by_k: dict[int, list[dict]],
    valid_features: list[str],
) -> list[str]:
    """
    Build a k-balanced machine-specific expansion pool.

    Round-robin extraction across k prevents one neighborhood size from
    dominating the higher-order search pool.
    """

    ordered = []

    def add(feature):
        if feature in valid_features and feature not in ordered:
            ordered.append(feature)

    single_by_k = {}
    for k in K_VALUES:
        rows_k = [r for r in single_rows if r["k"] == k]
        single_by_k[k] = sorted(rows_k, key=ranking_tuple, reverse=True)

    # Round-robin strongest singles across k.
    for rank_idx in range(TOP_SINGLE_PER_K_FOR_POOL):
        for k in K_VALUES:
            ranked = single_by_k[k]
            if rank_idx < len(ranked):
                add(ranked[rank_idx]["subset"][0])

    # Round-robin strongest pairs across k.
    for rank_idx in range(TOP_PAIR_PER_K_FOR_POOL):
        for k in K_VALUES:
            ranked = pair_top_by_k[k]
            if rank_idx < len(ranked):
                for feature in ranked[rank_idx]["subset"]:
                    add(feature)

                if (
                    not EXPAND_WITH_ALL_VALID_FEATURES
                    and MAX_EXPANSION_POOL is not None
                    and len(ordered) >= MAX_EXPANSION_POOL
                ):
                    break
        if (
            not EXPAND_WITH_ALL_VALID_FEATURES
            and MAX_EXPANSION_POOL is not None
            and len(ordered) >= MAX_EXPANSION_POOL
        ):
            break

    if EXPAND_WITH_ALL_VALID_FEATURES:
        for feature in valid_features:
            add(feature)
    elif MAX_EXPANSION_POOL is not None:
        ordered = ordered[:MAX_EXPANSION_POOL]

    return ordered


# =============================================================================
# MULTI-FEATURE k-PRESERVING BEAM SEARCH
# =============================================================================


def multi_feature_search(
    machine_name: str,
    data: dict,
    single_rows: list[dict],
    pair_top_by_k: dict[int, list[dict]],
    expansion_pool: list[str],
) -> tuple[list[dict], list[dict]]:
    """
    Search sizes 3..MAX_FEATURES.

    Critical design choice:
      top candidates are retained separately for every k and then unioned.

    Therefore a subset that is excellent for k=3 but mediocre for k=20 is
    not discarded just because another k currently has a stronger score.
    """

    best_by_size_k = []
    beam_history = []

    # Add exact size-1 best for every k.
    for k in K_VALUES:
        rows_k = [r for r in single_rows if r["k"] == k]
        ranked = sorted(rows_k, key=ranking_tuple, reverse=True)
        row = ranked[0].copy()
        row["search_stage"] = "exhaustive_single"
        best_by_size_k.append(row)

    # Add exact size-2 best for every k and create initial multi-k beam.
    beam_subsets = set()

    for k in K_VALUES:
        ranked = pair_top_by_k[k]
        if not ranked:
            raise RuntimeError(f"No pair results for {machine_name}, k={k}.")

        row = ranked[0].copy()
        row["search_stage"] = "exhaustive_pair"
        best_by_size_k.append(row)

        for rank, candidate in enumerate(ranked[:BEAM_PER_K], start=1):
            beam_subsets.add(candidate["subset"])

            hist = candidate.copy()
            hist["search_stage"] = "exhaustive_pair_seed"
            hist["beam_rank_for_k"] = rank
            beam_history.append(hist)

    # Sizes 3+.
    for target_size in range(3, MAX_FEATURES + 1):
        print(
            f"  Multi search size={target_size}: "
            f"parents={len(beam_subsets)}, pool={len(expansion_pool)}"
        )

        candidate_subsets = set()

        for base_subset in beam_subsets:
            base_set = set(base_subset)

            for feature in expansion_pool:
                if feature in base_set:
                    continue

                candidate = canonical_subset((*base_subset, feature))
                if len(candidate) != target_size:
                    continue

                if subset_is_redundant(candidate, data):
                    continue

                candidate_subsets.add(candidate)

        if not candidate_subsets:
            print(f"    no valid candidates at size={target_size}; stopping.")
            break

        keep_n = max(BEAM_PER_K, SAVE_TOP_PER_SIZE_PER_K)
        heaps = {k: [] for k in K_VALUES}

        total = len(candidate_subsets)
        progress_every = max(1000, total // 10)

        for candidate_no, subset in enumerate(candidate_subsets, start=1):
            rows = evaluate_subset_all_k(machine_name, subset, data)

            for row in rows:
                push_top(heaps[row["k"]], row, keep_n)

            if candidate_no % progress_every == 0 or candidate_no == total:
                print(f"    candidates {candidate_no:,}/{total:,}")

        new_beam_subsets = set()

        for k in K_VALUES:
            ranked = sorted_heap_rows(heaps[k])
            if not ranked:
                continue

            best = ranked[0].copy()
            best["search_stage"] = "multi_k_beam"
            best_by_size_k.append(best)

            print(
                f"    k={k:2d}  "
                f"objective={ranking_tuple(best)[0]:.4f}  "
                f"AUC={best['auc_all']:.4f}  "
                f"src={best['auc_source']:.4f}  "
                f"tgt={best['auc_target']:.4f}  "
                f"pAUC={best['pauc_01']:.4f}"
            )

            for rank, row in enumerate(ranked[:BEAM_PER_K], start=1):
                new_beam_subsets.add(row["subset"])

            for rank, row in enumerate(
                ranked[:SAVE_TOP_PER_SIZE_PER_K], start=1
            ):
                hist = row.copy()
                hist["search_stage"] = "multi_k_beam"
                hist["beam_rank_for_k"] = rank
                beam_history.append(hist)

        if not new_beam_subsets:
            break

        beam_subsets = new_beam_subsets

    return best_by_size_k, beam_history


# =============================================================================
# FINAL LOCAL SWAP REFINEMENT
# =============================================================================


def refine_best_for_k(
    machine_name: str,
    start_row: dict,
    data: dict,
    expansion_pool: list[str],
) -> dict:
    """Greedy same-size swap refinement for one fixed k."""

    if not ENABLE_FINAL_SWAP_REFINEMENT or start_row["n_features"] <= 2:
        return start_row

    target_k = int(start_row["k"])
    current = start_row.copy()
    local_cache = {}

    candidates_pool = expansion_pool[:SWAP_CANDIDATE_LIMIT]

    def evaluate_for_target_k(subset):
        subset = canonical_subset(subset)
        if subset not in local_cache:
            rows = evaluate_subset_all_k(machine_name, subset, data)
            local_cache[subset] = {row["k"]: row for row in rows}
        return local_cache[subset].get(target_k)

    for _pass in range(MAX_SWAP_PASSES):
        improved = False
        best_candidate = current
        current_subset = current["subset"]
        current_set = set(current_subset)

        for remove_feature in current_subset:
            reduced = current_set - {remove_feature}

            for add_feature in candidates_pool:
                if add_feature in reduced:
                    continue

                candidate_subset = canonical_subset((*reduced, add_feature))
                if len(candidate_subset) != len(current_subset):
                    continue

                if subset_is_redundant(candidate_subset, data):
                    continue

                row = evaluate_for_target_k(candidate_subset)
                if row is None:
                    continue

                if ranking_tuple(row) > ranking_tuple(best_candidate):
                    best_candidate = row
                    improved = True

        if not improved:
            break

        current = best_candidate.copy()

    current["search_stage"] = "final_swap_refinement"
    return current


# =============================================================================
# MACHINE SEARCH
# =============================================================================


def search_one_machine(machine_name: str, data: dict) -> dict:
    print("\n" + "=" * 100)
    print(f"MACHINE: {machine_name}")
    print("=" * 100)
    print(f"  usable TDA features = {len(data['feature_names'])}")
    print(f"  normal train samples = {len(data['X_train'])}")
    print(f"  labeled test samples = {len(data['X_test'])}")
    print(
        "  test domains = "
        + ", ".join(
            f"{d}:{int(np.sum(data['domains'] == d))}"
            for d in ["source", "target", "unknown"]
            if np.sum(data["domains"] == d) > 0
        )
    )

    # 1) Exact singles.
    single_rows = exhaustive_single_search(machine_name, data)

    # 2) Exact pairs.
    pair_top_by_k = exhaustive_pair_search(machine_name, data)

    # 3) Machine-specific higher-order pool, balanced across k.
    expansion_pool = build_expansion_pool(
        single_rows,
        pair_top_by_k,
        data["feature_names"],
    )

    print(f"  expansion pool size = {len(expansion_pool)}")

    # 4) k-preserving higher-order beam search.
    best_by_size_k, beam_history = multi_feature_search(
        machine_name,
        data,
        single_rows,
        pair_top_by_k,
        expansion_pool,
    )

    # 5) Best pre-refinement subset for each k across all subset sizes.
    pre_best_by_k = []
    for k in K_VALUES:
        rows_k = [row for row in best_by_size_k if row["k"] == k]
        pre_best_by_k.append(best_row(rows_k).copy())

    # 6) Final same-size local refinement for each k.
    refined_rows = []
    for row in pre_best_by_k:
        refined = refine_best_for_k(
            machine_name,
            row,
            data,
            expansion_pool,
        )
        refined_rows.append(refined)

        if ranking_tuple(refined) > ranking_tuple(row):
            # Add the refined candidate to size-wise candidates so final grouping
            # can correctly replace the previous best at that same size/k.
            best_by_size_k.append(refined)

    # 7) Final best machine x k.
    best_machine_k = []
    for k in K_VALUES:
        candidates = [row for row in best_by_size_k if row["k"] == k]
        best_machine_k.append(best_row(candidates).copy())

    # 8) Final single best subset+k for the machine.
    best_overall = best_row(best_machine_k).copy()

    # Flatten retained pair rows for export.
    pair_top_rows = []
    for k in K_VALUES:
        pair_top_rows.extend(pair_top_by_k[k])

    return {
        "single_rows": single_rows,
        "pair_top_rows": pair_top_rows,
        "best_by_size_k": best_by_size_k,
        "beam_history": beam_history,
        "expansion_pool": expansion_pool,
        "best_machine_k": best_machine_k,
        "best_overall": best_overall,
    }


# =============================================================================
# DATAFRAME / EXPORT HELPERS
# =============================================================================


def rows_to_dataframe(rows: list[dict]) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    if "subset" in df.columns:
        # Keep human-readable features column; tuple is not needed in Excel.
        df = df.drop(columns=["subset"])

    preferred = [
        "machine",
        "k",
        "n_features",
        "features",
        "dcase_hm",
        "auc_all",
        "auc_source",
        "auc_target",
        "pauc_01",
        "min_domain_auc",
        "search_stage",
        "beam_rank_for_k",
    ]

    cols = [c for c in preferred if c in df.columns] + [
        c for c in df.columns if c not in preferred
    ]

    return df[cols]


def select_best_group_rows(rows: list[dict], group_keys: list[str]) -> list[dict]:
    groups = {}
    for row in rows:
        key = tuple(row[k] for k in group_keys)
        previous = groups.get(key)
        if previous is None or ranking_tuple(row) > ranking_tuple(previous):
            groups[key] = row
    return list(groups.values())


def make_feature_stability(best_machine_k_rows: list[dict]) -> pd.DataFrame:
    machine_k_count = Counter()
    machine_sets = {}
    k_sets = {}

    for row in best_machine_k_rows:
        machine = row["machine"]
        k = int(row["k"])

        for feature in row["subset"]:
            machine_k_count[feature] += 1
            machine_sets.setdefault(feature, set()).add(machine)
            k_sets.setdefault(feature, set()).add(k)

    rows = []
    for feature in sorted(machine_k_count):
        rows.append(
            {
                "feature": feature,
                "selected_machine_k_count": machine_k_count[feature],
                "selected_machine_count": len(machine_sets[feature]),
                "selected_k_count": len(k_sets[feature]),
                "machines": " | ".join(sorted(machine_sets[feature])),
                "k_values": " | ".join(map(str, sorted(k_sets[feature]))),
            }
        )

    return pd.DataFrame(rows).sort_values(
        ["selected_machine_count", "selected_machine_k_count", "selected_k_count", "feature"],
        ascending=[False, False, False, True],
    )


def save_json_feature_pool(best_machine_k_rows, best_overall_rows, output_path: Path):
    feature_pool = sorted(
        {
            feature
            for row in best_machine_k_rows
            for feature in row["subset"]
        }
    )

    overall_pool = sorted(
        {
            feature
            for row in best_overall_rows
            for feature in row["subset"]
        }
    )

    feature_counts = Counter(
        feature
        for row in best_machine_k_rows
        for feature in row["subset"]
    )

    machine_results = {}
    for row in best_machine_k_rows:
        machine_results.setdefault(row["machine"], {})[str(row["k"])] = {
            "n_features": int(row["n_features"]),
            "features": list(row["subset"]),
            "dcase_hm": float(row["dcase_hm"]),
            "auc_all": float(row["auc_all"]),
            "auc_source": float(row["auc_source"]),
            "auc_target": float(row["auc_target"]),
            "pauc_01": float(row["pauc_01"]),
        }

    overall_results = {}
    for row in best_overall_rows:
        overall_results[row["machine"]] = {
            "best_k": int(row["k"]),
            "n_features": int(row["n_features"]),
            "features": list(row["subset"]),
            "dcase_hm": float(row["dcase_hm"]),
            "auc_all": float(row["auc_all"]),
            "auc_source": float(row["auc_source"]),
            "auc_target": float(row["auc_target"]),
            "pauc_01": float(row["pauc_01"]),
        }

    payload = {
        "description": (
            "Machine-specific supervised development oracle. Feature subset and "
            "k are selected using labeled development test data."
        ),
        "warning": (
            "This is an oracle/capability result, not a deployable unseen-machine "
            "feature-selection method."
        ),
        "score_direction": (
            "Higher mean kNN distance = more anomalous; scores are never inverted."
        ),
        "k_values": list(K_VALUES),
        "objective": OBJECTIVE,
        "objective_definition": (
            "HM(AUC_source, AUC_target, pAUC@0.1), then pAUC, "
            "min(source,target AUC), AUC_all"
            if OBJECTIVE == "dcase_hm"
            else "AUC_all, then pAUC"
        ),
        "n_unique_features_in_best_machine_k_solutions": len(feature_pool),
        "feature_pool_best_machine_k": feature_pool,
        "feature_selection_count_across_machine_k": dict(
            sorted(feature_counts.items(), key=lambda x: (-x[1], x[0]))
        ),
        "n_unique_features_in_best_overall_machine_solutions": len(overall_pool),
        "feature_pool_best_overall": overall_pool,
        "best_by_machine_k": machine_results,
        "best_overall_by_machine": overall_results,
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


# =============================================================================
# MAIN
# =============================================================================


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 100)
    print("JOINT k + MACHINE-SPECIFIC TDA ORACLE SEARCH")
    print("=" * 100)
    print(f"DATA_DIR       : {DATA_DIR}")
    print(f"K_VALUES       : {K_VALUES}")
    print(f"OBJECTIVE      : {OBJECTIVE}")
    print(f"MAX_FEATURES   : {MAX_FEATURES}")
    print(f"BEAM_PER_K     : {BEAM_PER_K}")
    print(f"OUTPUT_DIR     : {OUTPUT_DIR}")
    print()
    print("IMPORTANT: labeled development test data selects BOTH subset and k.")
    print("Anomaly score direction is fixed: larger kNN distance = anomaly.")

    datasets = detect_datasets(DATA_DIR)

    print("\nDetected complete machine pairs:")
    for key in sorted(datasets):
        info = datasets[key]
        print(
            f"  {info['display_name']:<15s} "
            f"train={info['train'].name}  test={info['test'].name}"
        )

    all_single_rows = []
    all_pair_top_rows = []
    all_best_by_size_k_rows = []
    all_beam_history_rows = []
    all_best_machine_k_rows = []
    all_best_overall_rows = []
    expansion_pool_rows = []

    for machine_no, machine_key in enumerate(sorted(datasets), start=1):
        info = datasets[machine_key]
        machine_name = info["display_name"]

        print(
            f"\n[{machine_no}/{len(datasets)}] Loading {machine_name} ..."
        )

        train_df = pd.read_excel(info["train"])
        test_df = pd.read_excel(info["test"])

        data = prepare_machine_data(train_df, test_df)
        result = search_one_machine(machine_name, data)

        all_single_rows.extend(result["single_rows"])
        all_pair_top_rows.extend(result["pair_top_rows"])
        all_best_by_size_k_rows.extend(result["best_by_size_k"])
        all_beam_history_rows.extend(result["beam_history"])
        all_best_machine_k_rows.extend(result["best_machine_k"])
        all_best_overall_rows.append(result["best_overall"])

        for pool_order, feature in enumerate(result["expansion_pool"], start=1):
            expansion_pool_rows.append(
                {
                    "machine": machine_name,
                    "pool_order": pool_order,
                    "feature": feature,
                }
            )

        print("\n  BEST BY k")
        for row in sorted(result["best_machine_k"], key=lambda r: r["k"]):
            print(
                f"    k={row['k']:2d}  n={row['n_features']:2d}  "
                f"HM={row['dcase_hm']:.4f}  AUC={row['auc_all']:.4f}  "
                f"src={row['auc_source']:.4f}  tgt={row['auc_target']:.4f}  "
                f"pAUC={row['pauc_01']:.4f}"
            )

        best = result["best_overall"]
        print("\n  OVERALL BEST subset + k")
        print(
            f"    k={best['k']}  n={best['n_features']}  "
            f"HM={best['dcase_hm']:.4f}  AUC={best['auc_all']:.4f}"
        )
        print(f"    {best['features']}")

    # -------------------------------------------------------------------------
    # Deduplicate/group results after optional refinements.
    # -------------------------------------------------------------------------

    best_by_size_k_final = select_best_group_rows(
        all_best_by_size_k_rows,
        ["machine", "k", "n_features"],
    )

    best_machine_k_final = select_best_group_rows(
        best_by_size_k_final,
        ["machine", "k"],
    )

    best_overall_final = select_best_group_rows(
        best_machine_k_final,
        ["machine"],
    )

    # Keep top single rows per machine/k for readable Excel size.
    top_single_export_rows = []
    single_groups = {}
    for row in all_single_rows:
        single_groups.setdefault((row["machine"], row["k"]), []).append(row)

    for key, rows in single_groups.items():
        ranked = sorted(rows, key=ranking_tuple, reverse=True)
        for rank, row in enumerate(ranked[:100], start=1):
            r = row.copy()
            r["rank_for_machine_k"] = rank
            top_single_export_rows.append(r)

    # Pair rows are already top-N per machine/k; add rank.
    top_pair_export_rows = []
    pair_groups = {}
    for row in all_pair_top_rows:
        pair_groups.setdefault((row["machine"], row["k"]), []).append(row)

    for key, rows in pair_groups.items():
        ranked = sorted(rows, key=ranking_tuple, reverse=True)
        for rank, row in enumerate(ranked, start=1):
            r = row.copy()
            r["rank_for_machine_k"] = rank
            top_pair_export_rows.append(r)

    feature_stability = make_feature_stability(best_machine_k_final)

    # -------------------------------------------------------------------------
    # Excel output
    # -------------------------------------------------------------------------

    best_overall_df = rows_to_dataframe(best_overall_final).sort_values("machine")
    best_machine_k_df = rows_to_dataframe(best_machine_k_final).sort_values(
        ["machine", "k"]
    )
    best_by_size_k_df = rows_to_dataframe(best_by_size_k_final).sort_values(
        ["machine", "k", "n_features"]
    )
    top_singles_df = rows_to_dataframe(top_single_export_rows).sort_values(
        ["machine", "k", "rank_for_machine_k"]
    )
    top_pairs_df = rows_to_dataframe(top_pair_export_rows).sort_values(
        ["machine", "k", "rank_for_machine_k"]
    )
    beam_history_df = rows_to_dataframe(all_beam_history_rows).sort_values(
        ["machine", "n_features", "k", "beam_rank_for_k"]
    )
    expansion_pool_df = pd.DataFrame(expansion_pool_rows).sort_values(
        ["machine", "pool_order"]
    )

    config_df = pd.DataFrame(
        {
            "parameter": [
                "K_VALUES",
                "OBJECTIVE",
                "PAUC_MAX_FPR",
                "MAX_FEATURES",
                "BEAM_PER_K",
                "TOP_SINGLE_PER_K_FOR_POOL",
                "TOP_PAIR_PER_K_FOR_POOL",
                "MAX_EXPANSION_POOL",
                "EXPAND_WITH_ALL_VALID_FEATURES",
                "USE_REDUNDANCY_FILTER",
                "MAX_ABS_TRAIN_CORR",
                "ENABLE_FINAL_SWAP_REFINEMENT",
                "SWAP_CANDIDATE_LIMIT",
                "MAX_SWAP_PASSES",
                "score_direction",
            ],
            "value": [
                str(K_VALUES),
                OBJECTIVE,
                PAUC_MAX_FPR,
                MAX_FEATURES,
                BEAM_PER_K,
                TOP_SINGLE_PER_K_FOR_POOL,
                TOP_PAIR_PER_K_FOR_POOL,
                MAX_EXPANSION_POOL,
                EXPAND_WITH_ALL_VALID_FEATURES,
                USE_REDUNDANCY_FILTER,
                MAX_ABS_TRAIN_CORR,
                ENABLE_FINAL_SWAP_REFINEMENT,
                SWAP_CANDIDATE_LIMIT,
                MAX_SWAP_PASSES,
                "higher mean kNN distance = anomaly; no score inversion",
            ],
        }
    )

    with pd.ExcelWriter(OUTPUT_XLSX) as writer:
        best_overall_df.to_excel(writer, sheet_name="Best_Overall", index=False)
        best_machine_k_df.to_excel(writer, sheet_name="Best_By_Machine_K", index=False)
        best_by_size_k_df.to_excel(writer, sheet_name="Best_By_Size_K", index=False)
        top_singles_df.to_excel(writer, sheet_name="Top_Singles", index=False)
        top_pairs_df.to_excel(writer, sheet_name="Top_Pairs", index=False)
        beam_history_df.to_excel(writer, sheet_name="Beam_History", index=False)
        expansion_pool_df.to_excel(writer, sheet_name="Expansion_Pool", index=False)
        feature_stability.to_excel(writer, sheet_name="Feature_Stability", index=False)
        config_df.to_excel(writer, sheet_name="Config", index=False)

    # -------------------------------------------------------------------------
    # JSON feature pool output
    # -------------------------------------------------------------------------

    save_json_feature_pool(
        best_machine_k_final,
        best_overall_final,
        OUTPUT_JSON,
    )

    # -------------------------------------------------------------------------
    # Final console summary
    # -------------------------------------------------------------------------

    print("\n" + "=" * 100)
    print("FINAL BEST subset + k PER MACHINE")
    print("=" * 100)

    display_cols = [
        "machine",
        "k",
        "n_features",
        "dcase_hm",
        "auc_all",
        "auc_source",
        "auc_target",
        "pauc_01",
        "features",
    ]

    print(best_overall_df[display_cols].round(4).to_string(index=False))

    print("\nSaved:")
    print(f"  {OUTPUT_XLSX}")
    print(f"  {OUTPUT_JSON}")


if __name__ == "__main__":
    main()

In [ ]:
# @title
# ============================================================
# GLOBAL TDA FEATURE SUBSET + GLOBAL k ORACLE SEARCH
# DCASE 2025 Task 2 - Development Set
#
# RUN THIS AFTER:
#   joint_k_machine_specific_tda_oracle.py
#
# GOAL
# ----
# Find ONE GLOBAL TDA feature subset S and ONE GLOBAL k value
# that are used unchanged for ALL development machines.
#
# This is a SUPERVISED DEVELOPMENT ORACLE / CAPABILITY STUDY:
# labeled development test data are intentionally used to search
# the global subset and global k.
#
# PRIMARY GLOBAL OBJECTIVE
# ------------------------
# For each machine m:
#
#   machine_score_m = HM(
#       AUC_source_m,
#       AUC_target_m,
#       pAUC@0.1_m
#   )
#
# For one common subset S and one common k:
#
#   GLOBAL_SCORE(S,k)
#       = HM(machine_score_1, ..., machine_score_M)
#
# Because every machine score is itself the HM of exactly 3
# components, this is equivalent to the harmonic mean of all
# machine x {source AUC, target AUC, pAUC} components.
#
# Tie-breakers:
#   1) mean machine DCASE HM
#   2) minimum machine DCASE HM
#   3) mean pAUC@0.1
#   4) mean min(AUC_source, AUC_target)
#   5) mean AUC(all)
#   6) fewer features
#   7) smaller k
#
# SEARCH
# ------
# Candidate feature pool:
#   by default uses the broader
#   feature_pool_best_machine_k
#   from best_oracle_feature_pool.json produced by the previous code.
#
# If unavailable:
#   falls back to feature_pool_best_overall,
#   then Best_Overall sheet,
#   then all common H0_/H1_ features.
#
# Search strategy:
#   size 1 : exhaustive over candidate pool
#   size 2 : exhaustive over all pairs
#   size 3..MAX_FEATURES:
#       global-k-preserving multi-beam forward search
#
# For EVERY subset evaluation:
#   - all machines are evaluated
#   - k in {3,5,10,20,30} is evaluated from the same neighbor query
#   - the SAME subset and SAME k are used across all machines
#
# Final:
#   same-size global swap refinement for the best subset of each k.
#
# OUTPUT
# ------
# /content/global_tda_subset_k_oracle/
#   global_tda_subset_k_oracle.xlsx
#   best_global_subset_k.json
#
# Excel sheets:
#   Best_Global
#   Best_By_Global_K
#   Best_By_Size_K
#   Per_Machine_Best_Global
#   Top_Global_Singles
#   Top_Global_Pairs
#   Beam_History
#   Candidate_Pool
#   Config
#
# IMPORTANT
# ---------
# Larger mean kNN distance = more anomalous.
# Scores are NEVER inverted.
# ============================================================

from __future__ import annotations

from pathlib import Path
from itertools import combinations, count
import heapq
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")


# =============================================================================
# CONFIGURATION
# =============================================================================

DATA_DIR = Path("/content")

# Output from the previous machine-specific oracle code.
PREVIOUS_OUTPUT_DIR = DATA_DIR / "joint_k_machine_specific_tda_oracle"
PREVIOUS_JSON = PREVIOUS_OUTPUT_DIR / "best_oracle_feature_pool.json"
PREVIOUS_XLSX = PREVIOUS_OUTPUT_DIR / "joint_k_machine_specific_tda_oracle.xlsx"

OUTPUT_DIR = DATA_DIR / "global_tda_subset_k_oracle"
OUTPUT_XLSX = OUTPUT_DIR / "global_tda_subset_k_oracle.xlsx"
OUTPUT_JSON = OUTPUT_DIR / "best_global_subset_k.json"

K_VALUES = (3, 5, 10, 20, 30)
MAX_K = max(K_VALUES)
PAUC_MAX_FPR = 0.10

# "best_machine_k" = broader union from the previous run.
# "best_overall"   = union of only the single best subset per machine.
POOL_MODE = "best_machine_k"

# Global subset search depth.
# Increase if you explicitly want to explore larger common representations.
MAX_FEATURES = 30

# Beam search for sizes >=3.
BEAM_PER_K = 15
SAVE_TOP_PER_SIZE_PER_K = 15

# Exact pair search retains these top rows per k.
PAIR_KEEP_TOP_PER_K = max(500, BEAM_PER_K)

# Higher-order expansion:
# If the previous-result pool is not too large, use all of it.
# Otherwise build a strong global expansion pool from singles+pairs.
MAX_EXPANSION_POOL = 120
TOP_SINGLE_PER_K_FOR_POOL = 40
TOP_PAIR_PER_K_FOR_POOL = 100

# Mild redundancy filter from NORMAL TRAIN only.
USE_REDUNDANCY_FILTER = True
MAX_ABS_TRAIN_CORR = 0.995

# Final same-size swap refinement for best global subset of each k.
ENABLE_FINAL_SWAP_REFINEMENT = True
SWAP_CANDIDATE_LIMIT = 100
MAX_SWAP_PASSES = 2

# cKDTree.
KD_WORKERS = -1

PAIR_PROGRESS_EVERY = 1000
MULTI_PROGRESS_EVERY = 500

# Cache all evaluated global subsets.
USE_EVALUATION_CACHE = True

EXCLUDED_TDA_FEATURES = {
    "H0_points_raw",
    "H1_points_raw",
}


# =============================================================================
# DATASET DETECTION
# =============================================================================

FILE_PATTERN = re.compile(
    r"^cubical_mel_tda_features_dev_(?P<machine>.+?)_"
    r"(?P<split>train|test)(?:\(\d+\))?$",
    flags=re.IGNORECASE,
)


def detect_datasets(data_dir: Path) -> dict:
    records = []

    for path in data_dir.rglob("*.xlsx"):
        # Do not accidentally read our output workbooks as datasets.
        if "joint_k_machine_specific_tda_oracle" in str(path):
            continue
        if "global_tda_subset_k_oracle" in str(path):
            continue

        match = FILE_PATTERN.match(path.stem)
        if not match:
            continue

        records.append(
            {
                "machine_raw": match.group("machine"),
                "machine_key": match.group("machine").lower(),
                "split": match.group("split").lower(),
                "path": path,
                "mtime": path.stat().st_mtime,
            }
        )

    if not records:
        raise FileNotFoundError(
            f"No DCASE development train/test Excel files found under {data_dir}."
        )

    newest = {}

    for rec in records:
        key = (rec["machine_key"], rec["split"])
        old = newest.get(key)

        if old is None or rec["mtime"] > old["mtime"]:
            newest[key] = rec

    datasets = {}

    for machine_key in sorted({k[0] for k in newest}):
        tr = newest.get((machine_key, "train"))
        te = newest.get((machine_key, "test"))

        if tr is None or te is None:
            print(
                f"WARNING: incomplete development pair ignored: {machine_key} "
                f"(train={tr is not None}, test={te is not None})"
            )
            continue

        datasets[machine_key] = {
            "display_name": tr["machine_raw"],
            "train": tr["path"],
            "test": te["path"],
        }

    if not datasets:
        raise FileNotFoundError("No complete development train/test pairs found.")

    return datasets


# =============================================================================
# LABEL / DOMAIN
# =============================================================================

def labels_to_binary(series: pd.Series) -> np.ndarray:
    if pd.api.types.is_numeric_dtype(series):
        values = pd.to_numeric(series, errors="coerce")

        if values.isna().any():
            raise ValueError("Some numeric labels cannot be interpreted.")

        return (values > 0).astype(np.int8).to_numpy()

    text = series.astype(str).str.strip().str.lower()

    anomaly_words = {
        "1",
        "anomaly",
        "anomalous",
        "abnormal",
        "fault",
        "faulty",
        "ng",
    }

    normal_words = {
        "0",
        "normal",
        "healthy",
        "ok",
    }

    out = []
    unknown = set()

    for value in text:
        if (
            value in anomaly_words
            or "anomal" in value
            or "abnormal" in value
            or "fault" in value
        ):
            out.append(1)

        elif (
            value in normal_words
            or "normal" in value
            or "healthy" in value
        ):
            out.append(0)

        else:
            out.append(-1)
            unknown.add(value)

    if unknown:
        raise ValueError(f"Unknown labels: {sorted(unknown)}")

    return np.asarray(out, dtype=np.int8)


def _normalize_domain_value(value) -> str:
    text = str(value).strip().lower()

    if "source" in text:
        return "source"

    if "target" in text:
        return "target"

    return "unknown"


def infer_test_domains(test_df: pd.DataFrame) -> np.ndarray:
    explicit_candidates = [
        "domain",
        "domain_label",
        "source_target",
        "data_domain",
        "dataset_domain",
    ]

    lower_to_real = {
        str(c).lower(): c
        for c in test_df.columns
    }

    for candidate in explicit_candidates:
        if candidate in lower_to_real:
            col = lower_to_real[candidate]

            domains = (
                test_df[col]
                .map(_normalize_domain_value)
                .to_numpy()
            )

            if np.mean(domains != "unknown") >= 0.95:
                return domains

    text_columns = []

    for candidate in [
        "file_id",
        "file_path",
        "filename",
        "path",
        "wav_path",
    ]:
        if candidate in lower_to_real:
            text_columns.append(lower_to_real[candidate])

    if not text_columns:
        return np.asarray(
            ["unknown"] * len(test_df),
            dtype=object,
        )

    combined = test_df[text_columns[0]].astype(str)

    for col in text_columns[1:]:
        combined = (
            combined
            + " "
            + test_df[col].astype(str)
        )

    return (
        combined
        .map(_normalize_domain_value)
        .to_numpy()
    )


# =============================================================================
# METRICS
# =============================================================================

def safe_auc(
    y: np.ndarray,
    scores: np.ndarray,
    max_fpr=None,
) -> float:
    if (
        len(y) == 0
        or len(np.unique(y)) < 2
    ):
        return float("nan")

    return float(
        roc_auc_score(
            y,
            scores,
            max_fpr=max_fpr,
        )
    )


def harmonic_mean(values) -> float:
    values = np.asarray(
        values,
        dtype=float,
    )

    if (
        len(values) == 0
        or np.any(~np.isfinite(values))
        or np.any(values <= 0)
    ):
        return float("nan")

    return float(
        len(values)
        / np.sum(
            1.0 / values
        )
    )


def compute_machine_metrics(
    y_test: np.ndarray,
    domains: np.ndarray,
    scores: np.ndarray,
) -> dict:
    auc_all = safe_auc(
        y_test,
        scores,
    )

    pauc_01 = safe_auc(
        y_test,
        scores,
        max_fpr=PAUC_MAX_FPR,
    )

    src_mask = domains == "source"
    tgt_mask = domains == "target"

    auc_source = safe_auc(
        y_test[src_mask],
        scores[src_mask],
    )

    auc_target = safe_auc(
        y_test[tgt_mask],
        scores[tgt_mask],
    )

    dcase_hm = harmonic_mean(
        [
            auc_source,
            auc_target,
            pauc_01,
        ]
    )

    min_domain_auc = float(
        min(
            auc_source,
            auc_target,
        )
    )

    return {
        "auc_all": auc_all,
        "auc_source": auc_source,
        "auc_target": auc_target,
        "pauc_01": pauc_01,
        "dcase_hm": dcase_hm,
        "min_domain_auc": min_domain_auc,
    }


def aggregate_global_metrics(
    machine_metrics: dict[str, dict],
) -> dict:
    rows = list(
        machine_metrics.values()
    )

    machine_hm = np.asarray(
        [
            r["dcase_hm"]
            for r in rows
        ],
        dtype=float,
    )

    auc_all = np.asarray(
        [
            r["auc_all"]
            for r in rows
        ],
        dtype=float,
    )

    auc_source = np.asarray(
        [
            r["auc_source"]
            for r in rows
        ],
        dtype=float,
    )

    auc_target = np.asarray(
        [
            r["auc_target"]
            for r in rows
        ],
        dtype=float,
    )

    pauc = np.asarray(
        [
            r["pauc_01"]
            for r in rows
        ],
        dtype=float,
    )

    min_domain = np.asarray(
        [
            r["min_domain_auc"]
            for r in rows
        ],
        dtype=float,
    )

    # Primary: harmonic mean across machine-level DCASE HM.
    global_dcase_hm = harmonic_mean(
        machine_hm
    )

    # Equivalent direct HM over the 3*M DCASE components.
    all_components = np.concatenate(
        [
            auc_source,
            auc_target,
            pauc,
        ]
    )

    direct_component_hm = harmonic_mean(
        all_components
    )

    return {
        "global_dcase_hm": global_dcase_hm,
        "global_component_hm": direct_component_hm,
        "mean_machine_dcase_hm": float(np.mean(machine_hm)),
        "min_machine_dcase_hm": float(np.min(machine_hm)),
        "mean_auc_all": float(np.mean(auc_all)),
        "mean_auc_source": float(np.mean(auc_source)),
        "mean_auc_target": float(np.mean(auc_target)),
        "mean_pauc_01": float(np.mean(pauc)),
        "mean_min_domain_auc": float(np.mean(min_domain)),
        "min_any_domain_auc": float(
            min(
                np.min(auc_source),
                np.min(auc_target),
            )
        ),
    }


def finite_or_neg_inf(value) -> float:
    try:
        value = float(value)
    except Exception:
        return -math.inf

    if not np.isfinite(value):
        return -math.inf

    return value


def global_ranking_tuple(row: dict) -> tuple:
    """
    Higher is better.

    Primary:
      global harmonic mean across all machine DCASE scores.

    Tie-breakers:
      mean machine HM
      worst machine HM
      mean pAUC
      mean min-domain AUC
      mean AUC(all)
      fewer features
      smaller k
    """
    return (
        finite_or_neg_inf(row["global_dcase_hm"]),
        finite_or_neg_inf(row["mean_machine_dcase_hm"]),
        finite_or_neg_inf(row["min_machine_dcase_hm"]),
        finite_or_neg_inf(row["mean_pauc_01"]),
        finite_or_neg_inf(row["mean_min_domain_auc"]),
        finite_or_neg_inf(row["mean_auc_all"]),
        -int(row["n_features"]),
        -int(row["k"]),
    )


# =============================================================================
# LOAD PREVIOUS FEATURE POOL
# =============================================================================

def split_feature_text(value) -> list[str]:
    if pd.isna(value):
        return []

    return [
        x.strip()
        for x in str(value).split("|")
        if x.strip()
    ]


def load_previous_feature_pool() -> tuple[list[str], str]:
    # Preferred: JSON produced by previous code.
    if PREVIOUS_JSON.exists():
        with open(
            PREVIOUS_JSON,
            "r",
            encoding="utf-8",
        ) as f:
            payload = json.load(f)

        if POOL_MODE == "best_machine_k":
            pool = payload.get(
                "feature_pool_best_machine_k",
                [],
            )

            if pool:
                return (
                    sorted(set(pool)),
                    "JSON: feature_pool_best_machine_k",
                )

        if POOL_MODE == "best_overall":
            pool = payload.get(
                "feature_pool_best_overall",
                [],
            )

            if pool:
                return (
                    sorted(set(pool)),
                    "JSON: feature_pool_best_overall",
                )

        # Fallback preference.
        for key in [
            "feature_pool_best_machine_k",
            "feature_pool_best_overall",
        ]:
            pool = payload.get(
                key,
                [],
            )

            if pool:
                return (
                    sorted(set(pool)),
                    f"JSON fallback: {key}",
                )

    # Second fallback: Best_Overall Excel sheet.
    if PREVIOUS_XLSX.exists():
        df = pd.read_excel(
            PREVIOUS_XLSX,
            sheet_name="Best_Overall",
        )

        if "features" in df.columns:
            pool = sorted(
                {
                    feature
                    for text in df["features"]
                    for feature in split_feature_text(text)
                }
            )

            if pool:
                return (
                    pool,
                    "Excel fallback: Best_Overall union",
                )

    return (
        [],
        "No previous pool found",
    )


# =============================================================================
# MACHINE PREPROCESSING
# =============================================================================

def prepare_machine_data(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> dict:
    if "label" not in test_df.columns:
        raise ValueError(
            "Development test file must contain label."
        )

    # Keep only normal train rows if train labels exist.
    if "label" in train_df.columns:
        try:
            y_train = labels_to_binary(
                train_df["label"]
            )

            if np.any(y_train == 1):
                train_df = (
                    train_df
                    .loc[y_train == 0]
                    .reset_index(drop=True)
                )
        except Exception:
            pass

    features = [
        c
        for c in train_df.columns
        if (
            c in test_df.columns
            and str(c).startswith(
                ("H0_", "H1_")
            )
            and c not in EXCLUDED_TDA_FEATURES
        )
    ]

    train_num = (
        train_df[features]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    test_num = (
        test_df[features]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    nonempty = [
        f
        for f in features
        if not train_num[f].isna().all()
    ]

    train_num = train_num[
        nonempty
    ]

    test_num = test_num[
        nonempty
    ]

    imputer = SimpleImputer(
        strategy="median"
    )

    X_train = imputer.fit_transform(
        train_num
    )

    X_test = imputer.transform(
        test_num
    )

    std = np.std(
        X_train,
        axis=0,
    )

    valid_mask = (
        np.isfinite(std)
        & (std > 1e-12)
    )

    feature_names = [
        f
        for f, keep
        in zip(
            nonempty,
            valid_mask,
        )
        if keep
    ]

    X_train = X_train[
        :,
        valid_mask,
    ]

    X_test = X_test[
        :,
        valid_mask,
    ]

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        X_train
    ).astype(
        np.float32
    )

    X_test = scaler.transform(
        X_test
    ).astype(
        np.float32
    )

    y_test = labels_to_binary(
        test_df["label"]
    )

    domains = infer_test_domains(
        test_df
    )

    for domain_name in [
        "source",
        "target",
    ]:
        mask = domains == domain_name

        if (
            np.sum(mask) == 0
            or len(
                np.unique(
                    y_test[mask]
                )
            ) < 2
        ):
            raise ValueError(
                f"Need both normal/anomaly in {domain_name} domain."
            )

    if len(X_train) < MAX_K:
        raise ValueError(
            f"Need >= {MAX_K} normal train rows."
        )

    feature_to_index = {
        f: i
        for i, f in enumerate(
            feature_names
        )
    }

    corr = np.corrcoef(
        X_train,
        rowvar=False,
    ).astype(
        np.float32
    )

    corr[
        ~np.isfinite(corr)
    ] = 0.0

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_test": y_test,
        "domains": domains,
        "feature_names": feature_names,
        "feature_to_index": feature_to_index,
        "corr": corr,
    }


# =============================================================================
# kNN
# =============================================================================

def query_knn_distances(
    tree: cKDTree,
    X_test: np.ndarray,
    k: int,
) -> np.ndarray:
    try:
        distances, _ = tree.query(
            X_test,
            k=k,
            workers=KD_WORKERS,
        )
    except TypeError:
        distances, _ = tree.query(
            X_test,
            k=k,
        )

    distances = np.asarray(
        distances,
        dtype=np.float64,
    )

    if distances.ndim == 1:
        distances = distances[
            :,
            None,
        ]

    return distances


def canonical_subset(features) -> tuple:
    return tuple(
        sorted(
            set(features)
        )
    )


# =============================================================================
# GLOBAL EVALUATOR
# =============================================================================

class GlobalSubsetEvaluator:
    def __init__(
        self,
        machine_data: dict[str, dict],
    ):
        self.machine_data = machine_data
        self.cache = {}

    def evaluate(
        self,
        subset,
    ) -> list[dict]:
        subset = canonical_subset(
            subset
        )

        if (
            USE_EVALUATION_CACHE
            and subset in self.cache
        ):
            return self.cache[subset]

        # scores_by_machine_k[machine][k] = metrics
        metrics_by_machine_k = {
            machine: {}
            for machine in self.machine_data
        }

        for machine, data in self.machine_data.items():
            index_map = data[
                "feature_to_index"
            ]

            if not all(
                f in index_map
                for f in subset
            ):
                return []

            indices = [
                index_map[f]
                for f in subset
            ]

            X_train = data[
                "X_train"
            ][
                :,
                indices,
            ]

            X_test = data[
                "X_test"
            ][
                :,
                indices,
            ]

            tree = cKDTree(
                X_train
            )

            distances = query_knn_distances(
                tree,
                X_test,
                MAX_K,
            )

            cumulative = np.cumsum(
                distances,
                axis=1,
            )

            for k in K_VALUES:
                scores = (
                    cumulative[
                        :,
                        k - 1,
                    ]
                    / float(k)
                )

                metrics_by_machine_k[
                    machine
                ][
                    k
                ] = (
                    compute_machine_metrics(
                        data[
                            "y_test"
                        ],
                        data[
                            "domains"
                        ],
                        scores,
                    )
                )

        rows = []

        for k in K_VALUES:
            machine_metrics = {
                machine: metrics_by_machine_k[
                    machine
                ][
                    k
                ]
                for machine
                in self.machine_data
            }

            global_metrics = aggregate_global_metrics(
                machine_metrics
            )

            row = {
                "k": int(k),
                "n_features": len(subset),
                "subset": subset,
                "features": " | ".join(subset),
                **global_metrics,
                "machine_metrics": machine_metrics,
            }

            rows.append(
                row
            )

        if USE_EVALUATION_CACHE:
            self.cache[
                subset
            ] = rows

        return rows


# =============================================================================
# GLOBAL REDUNDANCY FILTER
# =============================================================================

def subset_is_redundant_global(
    subset,
    machine_data: dict[str, dict],
) -> bool:
    if (
        not USE_REDUNDANCY_FILTER
        or len(subset) < 2
    ):
        return False

    # Reject only if the candidate contains an almost duplicate pair
    # in EVERY machine. This is deliberately mild.
    redundant_in_all = True

    for data in machine_data.values():
        idx = [
            data[
                "feature_to_index"
            ][f]
            for f in subset
        ]

        subcorr = np.abs(
            data[
                "corr"
            ][
                np.ix_(
                    idx,
                    idx,
                )
            ]
        )

        upper = subcorr[
            np.triu_indices(
                len(idx),
                k=1,
            )
        ]

        machine_redundant = bool(
            np.any(
                upper
                >= MAX_ABS_TRAIN_CORR
            )
        )

        if not machine_redundant:
            redundant_in_all = False
            break

    return redundant_in_all


# =============================================================================
# TOP-N HEAPS
# =============================================================================

_heap_counter = count()


def push_top(
    heap: list,
    row: dict,
    limit: int,
):
    item = (
        global_ranking_tuple(row),
        next(_heap_counter),
        row,
    )

    if len(heap) < limit:
        heapq.heappush(
            heap,
            item,
        )

    elif item[0] > heap[0][0]:
        heapq.heapreplace(
            heap,
            item,
        )


def sorted_heap_rows(
    heap: list,
) -> list[dict]:
    return sorted(
        [
            x[2]
            for x in heap
        ],
        key=global_ranking_tuple,
        reverse=True,
    )


def best_row(
    rows: list[dict],
) -> dict:
    if not rows:
        raise ValueError(
            "No rows available."
        )

    return max(
        rows,
        key=global_ranking_tuple,
    )


# =============================================================================
# SEARCH: SINGLES
# =============================================================================

def exhaustive_global_singles(
    candidate_pool: list[str],
    evaluator: GlobalSubsetEvaluator,
) -> list[dict]:
    print(
        "\nGLOBAL EXHAUSTIVE SINGLE-FEATURE SEARCH"
    )

    rows = []

    for i, feature in enumerate(
        candidate_pool,
        start=1,
    ):
        rows.extend(
            evaluator.evaluate(
                (feature,)
            )
        )

        if (
            i % 20 == 0
            or i == len(
                candidate_pool
            )
        ):
            print(
                f"  singles {i}/"
                f"{len(candidate_pool)}"
            )

    return rows


# =============================================================================
# SEARCH: PAIRS
# =============================================================================

def exhaustive_global_pairs(
    candidate_pool: list[str],
    evaluator: GlobalSubsetEvaluator,
    machine_data: dict[str, dict],
) -> dict[int, list[dict]]:
    n_pairs = (
        len(candidate_pool)
        * (
            len(candidate_pool)
            - 1
        )
        // 2
    )

    print(
        "\nGLOBAL EXHAUSTIVE PAIR SEARCH:"
        f" {n_pairs:,} pairs"
    )

    heaps = {
        k: []
        for k in K_VALUES
    }

    evaluated = 0

    for pair_no, pair in enumerate(
        combinations(
            candidate_pool,
            2,
        ),
        start=1,
    ):
        if subset_is_redundant_global(
            pair,
            machine_data,
        ):
            continue

        rows = evaluator.evaluate(
            pair
        )

        evaluated += 1

        for row in rows:
            push_top(
                heaps[
                    row["k"]
                ],
                row,
                PAIR_KEEP_TOP_PER_K,
            )

        if (
            pair_no
            % PAIR_PROGRESS_EVERY
            == 0
            or pair_no == n_pairs
        ):
            print(
                f"  pair positions "
                f"{pair_no:,}/{n_pairs:,} "
                f"| evaluated={evaluated:,}"
            )

    return {
        k: sorted_heap_rows(
            heaps[k]
        )
        for k in K_VALUES
    }


# =============================================================================
# EXPANSION POOL
# =============================================================================

def build_global_expansion_pool(
    candidate_pool: list[str],
    single_rows: list[dict],
    pair_top_by_k: dict[int, list[dict]],
) -> list[str]:
    if (
        MAX_EXPANSION_POOL is None
        or len(candidate_pool)
        <= MAX_EXPANSION_POOL
    ):
        return list(
            candidate_pool
        )

    ordered = []

    def add(feature):
        if (
            feature in candidate_pool
            and feature not in ordered
        ):
            ordered.append(
                feature
            )

    single_by_k = {}

    for k in K_VALUES:
        ranked = sorted(
            [
                row
                for row in single_rows
                if row["k"] == k
            ],
            key=global_ranking_tuple,
            reverse=True,
        )

        single_by_k[
            k
        ] = ranked

    # k-balanced strong global singles.
    for rank_idx in range(
        TOP_SINGLE_PER_K_FOR_POOL
    ):
        for k in K_VALUES:
            ranked = single_by_k[
                k
            ]

            if rank_idx < len(ranked):
                add(
                    ranked[
                        rank_idx
                    ][
                        "subset"
                    ][
                        0
                    ]
                )

    # k-balanced strong global pairs.
    for rank_idx in range(
        TOP_PAIR_PER_K_FOR_POOL
    ):
        for k in K_VALUES:
            ranked = pair_top_by_k[
                k
            ]

            if rank_idx < len(ranked):
                for feature in ranked[
                    rank_idx
                ][
                    "subset"
                ]:
                    add(
                        feature
                    )

                    if (
                        len(ordered)
                        >= MAX_EXPANSION_POOL
                    ):
                        return ordered[
                            :MAX_EXPANSION_POOL
                        ]

    # Fill with remaining prior-pool features.
    for feature in candidate_pool:
        add(
            feature
        )

        if (
            len(ordered)
            >= MAX_EXPANSION_POOL
        ):
            break

    return ordered[
        :MAX_EXPANSION_POOL
    ]


# =============================================================================
# MULTI-FEATURE GLOBAL BEAM SEARCH
# =============================================================================

def global_multi_feature_search(
    single_rows: list[dict],
    pair_top_by_k: dict[int, list[dict]],
    expansion_pool: list[str],
    evaluator: GlobalSubsetEvaluator,
    machine_data: dict[str, dict],
) -> tuple[list[dict], list[dict]]:
    best_by_size_k = []
    beam_history = []

    # Size 1 best for each global k.
    for k in K_VALUES:
        rows_k = [
            r
            for r in single_rows
            if r["k"] == k
        ]

        ranked = sorted(
            rows_k,
            key=global_ranking_tuple,
            reverse=True,
        )

        best = ranked[0].copy()
        best[
            "search_stage"
        ] = "exhaustive_single"

        best_by_size_k.append(
            best
        )

    # Size 2 best and seed beam.
    beam_subsets = set()

    for k in K_VALUES:
        ranked = pair_top_by_k[
            k
        ]

        if not ranked:
            raise RuntimeError(
                f"No global pair candidates for k={k}."
            )

        best = ranked[0].copy()
        best[
            "search_stage"
        ] = "exhaustive_pair"

        best_by_size_k.append(
            best
        )

        for rank, row in enumerate(
            ranked[
                :BEAM_PER_K
            ],
            start=1,
        ):
            beam_subsets.add(
                row["subset"]
            )

            hist = row.copy()
            hist[
                "search_stage"
            ] = "exhaustive_pair_seed"
            hist[
                "beam_rank_for_k"
            ] = rank

            beam_history.append(
                hist
            )

    max_size = min(
        MAX_FEATURES,
        len(expansion_pool),
    )

    for target_size in range(
        3,
        max_size + 1,
    ):
        print(
            f"\nGLOBAL MULTI SEARCH size={target_size} "
            f"| parents={len(beam_subsets)} "
            f"| expansion_pool={len(expansion_pool)}"
        )

        candidate_subsets = set()

        for base in beam_subsets:
            base_set = set(
                base
            )

            for feature in expansion_pool:
                if feature in base_set:
                    continue

                candidate = canonical_subset(
                    (
                        *base,
                        feature,
                    )
                )

                if len(candidate) != target_size:
                    continue

                if subset_is_redundant_global(
                    candidate,
                    machine_data,
                ):
                    continue

                candidate_subsets.add(
                    candidate
                )

        if not candidate_subsets:
            print(
                "  no candidates; stopping."
            )
            break

        keep_n = max(
            BEAM_PER_K,
            SAVE_TOP_PER_SIZE_PER_K,
        )

        heaps = {
            k: []
            for k in K_VALUES
        }

        total = len(
            candidate_subsets
        )

        for candidate_no, subset in enumerate(
            candidate_subsets,
            start=1,
        ):
            rows = evaluator.evaluate(
                subset
            )

            for row in rows:
                push_top(
                    heaps[
                        row["k"]
                    ],
                    row,
                    keep_n,
                )

            if (
                candidate_no
                % MULTI_PROGRESS_EVERY
                == 0
                or candidate_no == total
            ):
                print(
                    f"  candidates "
                    f"{candidate_no:,}/{total:,}"
                )

        new_beam = set()

        for k in K_VALUES:
            ranked = sorted_heap_rows(
                heaps[k]
            )

            if not ranked:
                continue

            best = ranked[
                0
            ].copy()

            best[
                "search_stage"
            ] = "global_multi_k_beam"

            best_by_size_k.append(
                best
            )

            print(
                f"    k={k:2d} "
                f"GLOBAL_HM="
                f"{best['global_dcase_hm']:.4f} "
                f"meanHM="
                f"{best['mean_machine_dcase_hm']:.4f} "
                f"worstHM="
                f"{best['min_machine_dcase_hm']:.4f} "
                f"meanAUC="
                f"{best['mean_auc_all']:.4f}"
            )

            for rank, row in enumerate(
                ranked[
                    :BEAM_PER_K
                ],
                start=1,
            ):
                new_beam.add(
                    row["subset"]
                )

            for rank, row in enumerate(
                ranked[
                    :SAVE_TOP_PER_SIZE_PER_K
                ],
                start=1,
            ):
                hist = row.copy()
                hist[
                    "search_stage"
                ] = "global_multi_k_beam"
                hist[
                    "beam_rank_for_k"
                ] = rank

                beam_history.append(
                    hist
                )

        if not new_beam:
            break

        beam_subsets = new_beam

    return (
        best_by_size_k,
        beam_history,
    )


# =============================================================================
# FINAL GLOBAL SWAP REFINEMENT
# =============================================================================

def refine_global_best_for_k(
    start_row: dict,
    evaluator: GlobalSubsetEvaluator,
    expansion_pool: list[str],
    machine_data: dict[str, dict],
) -> dict:
    if (
        not ENABLE_FINAL_SWAP_REFINEMENT
        or start_row[
            "n_features"
        ] <= 1
    ):
        return start_row

    target_k = int(
        start_row["k"]
    )

    current = start_row.copy()

    # Use strongest available expansion features.
    candidates_pool = expansion_pool[
        :SWAP_CANDIDATE_LIMIT
    ]

    for pass_no in range(
        1,
        MAX_SWAP_PASSES + 1,
    ):
        print(
            f"  swap refinement k={target_k}, "
            f"pass={pass_no}"
        )

        improved = False
        best_candidate = current

        current_subset = current[
            "subset"
        ]

        current_set = set(
            current_subset
        )

        for remove_feature in current_subset:
            reduced = (
                current_set
                - {
                    remove_feature
                }
            )

            for add_feature in candidates_pool:
                if add_feature in reduced:
                    continue

                candidate = canonical_subset(
                    (
                        *reduced,
                        add_feature,
                    )
                )

                if (
                    len(candidate)
                    != len(
                        current_subset
                    )
                ):
                    continue

                if subset_is_redundant_global(
                    candidate,
                    machine_data,
                ):
                    continue

                rows = evaluator.evaluate(
                    candidate
                )

                rows_k = [
                    r
                    for r in rows
                    if r[
                        "k"
                    ] == target_k
                ]

                if not rows_k:
                    continue

                row = rows_k[
                    0
                ]

                if (
                    global_ranking_tuple(
                        row
                    )
                    > global_ranking_tuple(
                        best_candidate
                    )
                ):
                    best_candidate = row
                    improved = True

        if not improved:
            break

        current = best_candidate.copy()

        print(
            "    improved -> "
            f"GLOBAL_HM="
            f"{current['global_dcase_hm']:.5f}"
        )

    current[
        "search_stage"
    ] = "final_global_swap_refinement"

    return current


# =============================================================================
# EXPORT HELPERS
# =============================================================================

def strip_machine_metrics(
    row: dict,
) -> dict:
    out = {
        k: v
        for k, v in row.items()
        if k not in {
            "subset",
            "machine_metrics",
        }
    }

    return out


def rows_to_dataframe(
    rows: list[dict],
) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame()

    clean = [
        strip_machine_metrics(
            row
        )
        for row in rows
    ]

    df = pd.DataFrame(
        clean
    )

    preferred = [
        "k",
        "n_features",
        "features",
        "global_dcase_hm",
        "global_component_hm",
        "mean_machine_dcase_hm",
        "min_machine_dcase_hm",
        "mean_auc_all",
        "mean_auc_source",
        "mean_auc_target",
        "mean_pauc_01",
        "mean_min_domain_auc",
        "min_any_domain_auc",
        "search_stage",
        "beam_rank_for_k",
    ]

    columns = (
        [
            c
            for c in preferred
            if c in df.columns
        ]
        + [
            c
            for c in df.columns
            if c not in preferred
        ]
    )

    return df[
        columns
    ]


def best_group_rows(
    rows: list[dict],
    keys: list[str],
) -> list[dict]:
    groups = {}

    for row in rows:
        key = tuple(
            row[
                k
            ]
            for k in keys
        )

        old = groups.get(
            key
        )

        if (
            old is None
            or global_ranking_tuple(
                row
            )
            > global_ranking_tuple(
                old
            )
        ):
            groups[
                key
            ] = row

    return list(
        groups.values()
    )


def per_machine_dataframe(
    global_row: dict,
) -> pd.DataFrame:
    rows = []

    for machine, metrics in global_row[
        "machine_metrics"
    ].items():
        rows.append(
            {
                "machine": machine,
                "global_k": global_row[
                    "k"
                ],
                "global_n_features": global_row[
                    "n_features"
                ],
                "dcase_hm": metrics[
                    "dcase_hm"
                ],
                "auc_all": metrics[
                    "auc_all"
                ],
                "auc_source": metrics[
                    "auc_source"
                ],
                "auc_target": metrics[
                    "auc_target"
                ],
                "pauc_01": metrics[
                    "pauc_01"
                ],
                "min_domain_auc": metrics[
                    "min_domain_auc"
                ],
                "features": global_row[
                    "features"
                ],
            }
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "machine"
        )
        .reset_index(
            drop=True
        )
    )


# =============================================================================
# MAIN
# =============================================================================

def main():
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "=" * 110
    )
    print(
        "GLOBAL TDA FEATURE SUBSET + GLOBAL k ORACLE SEARCH"
    )
    print(
        "=" * 110
    )

    print(
        f"DATA_DIR             : {DATA_DIR}"
    )
    print(
        f"PREVIOUS_JSON        : {PREVIOUS_JSON}"
    )
    print(
        f"POOL_MODE            : {POOL_MODE}"
    )
    print(
        f"K_VALUES             : {K_VALUES}"
    )
    print(
        f"MAX_FEATURES         : {MAX_FEATURES}"
    )
    print(
        f"BEAM_PER_K           : {BEAM_PER_K}"
    )
    print(
        f"MAX_EXPANSION_POOL   : {MAX_EXPANSION_POOL}"
    )
    print(
        f"OUTPUT_DIR           : {OUTPUT_DIR}"
    )
    print()
    print(
        "IMPORTANT: ONE COMMON subset + ONE COMMON k across ALL machines."
    )
    print(
        "IMPORTANT: labeled development test data are used; this is an oracle study."
    )
    print(
        "IMPORTANT: score direction is fixed; larger kNN distance = anomaly."
    )

    # -------------------------------------------------------------------------
    # Detect/load development machines.
    # -------------------------------------------------------------------------
    datasets = detect_datasets(
        DATA_DIR
    )

    print(
        "\nDetected development machines:"
    )

    for key in sorted(
        datasets
    ):
        info = datasets[
            key
        ]

        print(
            f"  {info['display_name']:<15s} "
            f"train={info['train'].name} "
            f"test={info['test'].name}"
        )

    machine_data = {}

    for key in sorted(
        datasets
    ):
        info = datasets[
            key
        ]

        machine = info[
            "display_name"
        ]

        print(
            f"\nLoading/preparing {machine} ..."
        )

        train_df = pd.read_excel(
            info[
                "train"
            ]
        )

        test_df = pd.read_excel(
            info[
                "test"
            ]
        )

        machine_data[
            machine
        ] = prepare_machine_data(
            train_df,
            test_df,
        )

        d = machine_data[
            machine
        ]

        print(
            f"  usable features={len(d['feature_names'])} "
            f"normal train={len(d['X_train'])} "
            f"test={len(d['X_test'])}"
        )

    # -------------------------------------------------------------------------
    # Previous result pool.
    # -------------------------------------------------------------------------
    prior_pool, pool_source = (
        load_previous_feature_pool()
    )

    common_features = set.intersection(
        *[
            set(
                data[
                    "feature_names"
                ]
            )
            for data
            in machine_data.values()
        ]
    )

    if prior_pool:
        candidate_pool = sorted(
            set(
                prior_pool
            )
            & common_features
        )

    else:
        candidate_pool = sorted(
            common_features
        )

        pool_source = (
            "Fallback: all common valid TDA features"
        )

    if len(
        candidate_pool
    ) < 2:
        raise ValueError(
            "Fewer than 2 common candidate features."
        )

    print(
        "\nCandidate pool:"
    )
    print(
        f"  source     : {pool_source}"
    )
    print(
        f"  prior count: {len(prior_pool)}"
    )
    print(
        f"  common used: {len(candidate_pool)}"
    )

    for i, feature in enumerate(
        candidate_pool,
        start=1,
    ):
        print(
            f"  {i:3d}. {feature}"
        )

    evaluator = GlobalSubsetEvaluator(
        machine_data
    )

    # -------------------------------------------------------------------------
    # 1) Exhaustive singles.
    # -------------------------------------------------------------------------
    single_rows = exhaustive_global_singles(
        candidate_pool,
        evaluator,
    )

    # -------------------------------------------------------------------------
    # 2) Exhaustive pairs.
    # -------------------------------------------------------------------------
    pair_top_by_k = exhaustive_global_pairs(
        candidate_pool,
        evaluator,
        machine_data,
    )

    # -------------------------------------------------------------------------
    # 3) Build higher-order expansion pool.
    # -------------------------------------------------------------------------
    expansion_pool = build_global_expansion_pool(
        candidate_pool,
        single_rows,
        pair_top_by_k,
    )

    print(
        "\nGlobal higher-order expansion pool:"
    )
    print(
        f"  size={len(expansion_pool)}"
    )

    # -------------------------------------------------------------------------
    # 4) Global k-preserving beam search.
    # -------------------------------------------------------------------------
    (
        best_by_size_k_rows,
        beam_history_rows,
    ) = global_multi_feature_search(
        single_rows,
        pair_top_by_k,
        expansion_pool,
        evaluator,
        machine_data,
    )

    # -------------------------------------------------------------------------
    # 5) Best pre-refinement per global k.
    # -------------------------------------------------------------------------
    pre_best_by_k = []

    for k in K_VALUES:
        rows_k = [
            row
            for row in best_by_size_k_rows
            if row[
                "k"
            ] == k
        ]

        pre_best_by_k.append(
            best_row(
                rows_k
            ).copy()
        )

    # -------------------------------------------------------------------------
    # 6) Final swap refinement per global k.
    # -------------------------------------------------------------------------
    refined_by_k = []

    print(
        "\nFINAL GLOBAL SAME-SIZE SWAP REFINEMENT"
    )

    for row in pre_best_by_k:
        refined = refine_global_best_for_k(
            row,
            evaluator,
            expansion_pool,
            machine_data,
        )

        refined_by_k.append(
            refined
        )

        # Preserve refined row in size-wise history.
        if (
            global_ranking_tuple(
                refined
            )
            > global_ranking_tuple(
                row
            )
        ):
            best_by_size_k_rows.append(
                refined
            )

    # -------------------------------------------------------------------------
    # 7) Final best by global k and overall.
    # -------------------------------------------------------------------------
    best_by_size_k_final = best_group_rows(
        best_by_size_k_rows,
        [
            "k",
            "n_features",
        ],
    )

    best_by_k_final = best_group_rows(
        (
            best_by_size_k_final
            + refined_by_k
        ),
        [
            "k",
        ],
    )

    best_global = best_row(
        best_by_k_final
    ).copy()

    # -------------------------------------------------------------------------
    # Console results.
    # -------------------------------------------------------------------------
    print(
        "\n"
        + "=" * 110
    )
    print(
        "BEST GLOBAL SUBSET FOR EACH GLOBAL k"
    )
    print(
        "=" * 110
    )

    for row in sorted(
        best_by_k_final,
        key=lambda r: r[
            "k"
        ],
    ):
        print(
            f"k={row['k']:2d} "
            f"n={row['n_features']:2d} "
            f"GLOBAL_HM={row['global_dcase_hm']:.4f} "
            f"meanHM={row['mean_machine_dcase_hm']:.4f} "
            f"worstHM={row['min_machine_dcase_hm']:.4f} "
            f"meanAUC={row['mean_auc_all']:.4f} "
            f"meanSrc={row['mean_auc_source']:.4f} "
            f"meanTgt={row['mean_auc_target']:.4f} "
            f"meanpAUC={row['mean_pauc_01']:.4f}"
        )

    print(
        "\n"
        + "=" * 110
    )
    print(
        "OVERALL BEST GLOBAL FEATURE SUBSET + GLOBAL k"
    )
    print(
        "=" * 110
    )

    print(
        f"GLOBAL k             = {best_global['k']}"
    )
    print(
        f"N features           = {best_global['n_features']}"
    )
    print(
        f"GLOBAL DCASE HM       = {best_global['global_dcase_hm']:.6f}"
    )
    print(
        f"GLOBAL component HM   = {best_global['global_component_hm']:.6f}"
    )
    print(
        f"Mean machine HM       = {best_global['mean_machine_dcase_hm']:.6f}"
    )
    print(
        f"Worst machine HM      = {best_global['min_machine_dcase_hm']:.6f}"
    )
    print(
        f"Mean AUC(all)         = {best_global['mean_auc_all']:.6f}"
    )
    print(
        f"Mean AUC(source)      = {best_global['mean_auc_source']:.6f}"
    )
    print(
        f"Mean AUC(target)      = {best_global['mean_auc_target']:.6f}"
    )
    print(
        f"Mean pAUC@0.1         = {best_global['mean_pauc_01']:.6f}"
    )
    print(
        f"Min any domain AUC    = {best_global['min_any_domain_auc']:.6f}"
    )

    print(
        "\nGLOBAL FEATURES:"
    )

    for i, f in enumerate(
        best_global[
            "subset"
        ],
        start=1,
    ):
        print(
            f"  {i:2d}. {f}"
        )

    per_machine_df = per_machine_dataframe(
        best_global
    )

    print(
        "\nPER-MACHINE PERFORMANCE OF THE SAME GLOBAL SUBSET + SAME GLOBAL k"
    )

    print(
        per_machine_df[
            [
                "machine",
                "dcase_hm",
                "auc_all",
                "auc_source",
                "auc_target",
                "pauc_01",
            ]
        ]
        .round(4)
        .to_string(
            index=False
        )
    )

    # -------------------------------------------------------------------------
    # Export top singles and pairs.
    # -------------------------------------------------------------------------
    top_single_rows = []

    for k in K_VALUES:
        ranked = sorted(
            [
                r
                for r in single_rows
                if r[
                    "k"
                ] == k
            ],
            key=global_ranking_tuple,
            reverse=True,
        )

        for rank, row in enumerate(
            ranked[
                :100
            ],
            start=1,
        ):
            r = row.copy()
            r[
                "rank_for_global_k"
            ] = rank

            top_single_rows.append(
                r
            )

    top_pair_rows = []

    for k in K_VALUES:
        for rank, row in enumerate(
            pair_top_by_k[
                k
            ],
            start=1,
        ):
            r = row.copy()
            r[
                "rank_for_global_k"
            ] = rank

            top_pair_rows.append(
                r
            )

    # -------------------------------------------------------------------------
    # Excel.
    # -------------------------------------------------------------------------
    best_global_df = rows_to_dataframe(
        [
            best_global
        ]
    )

    best_by_k_df = rows_to_dataframe(
        sorted(
            best_by_k_final,
            key=lambda r: r[
                "k"
            ],
        )
    )

    best_by_size_k_df = rows_to_dataframe(
        sorted(
            best_by_size_k_final,
            key=lambda r: (
                r[
                    "n_features"
                ],
                r[
                    "k"
                ],
            ),
        )
    )

    top_singles_df = rows_to_dataframe(
        top_single_rows
    )

    top_pairs_df = rows_to_dataframe(
        top_pair_rows
    )

    beam_df = rows_to_dataframe(
        beam_history_rows
    )

    candidate_pool_df = pd.DataFrame(
        {
            "pool_order": np.arange(
                1,
                len(
                    candidate_pool
                )
                + 1,
            ),
            "feature": candidate_pool,
            "in_expansion_pool": [
                f in set(
                    expansion_pool
                )
                for f in candidate_pool
            ],
        }
    )

    config_df = pd.DataFrame(
        {
            "parameter": [
                "POOL_MODE",
                "POOL_SOURCE",
                "N_CANDIDATE_FEATURES",
                "K_VALUES",
                "MAX_FEATURES",
                "BEAM_PER_K",
                "SAVE_TOP_PER_SIZE_PER_K",
                "MAX_EXPANSION_POOL",
                "USE_REDUNDANCY_FILTER",
                "MAX_ABS_TRAIN_CORR",
                "ENABLE_FINAL_SWAP_REFINEMENT",
                "SWAP_CANDIDATE_LIMIT",
                "MAX_SWAP_PASSES",
                "PRIMARY_GLOBAL_OBJECTIVE",
                "score_direction",
                "oracle_warning",
            ],
            "value": [
                POOL_MODE,
                pool_source,
                len(
                    candidate_pool
                ),
                str(
                    K_VALUES
                ),
                MAX_FEATURES,
                BEAM_PER_K,
                SAVE_TOP_PER_SIZE_PER_K,
                MAX_EXPANSION_POOL,
                USE_REDUNDANCY_FILTER,
                MAX_ABS_TRAIN_CORR,
                ENABLE_FINAL_SWAP_REFINEMENT,
                SWAP_CANDIDATE_LIMIT,
                MAX_SWAP_PASSES,
                (
                    "HM of machine-level "
                    "HM(AUC_source,AUC_target,pAUC@0.1)"
                ),
                (
                    "higher mean kNN distance = anomaly; "
                    "no score inversion"
                ),
                (
                    "supervised development oracle; "
                    "not a deployable unseen selector"
                ),
            ],
        }
    )

    with pd.ExcelWriter(
        OUTPUT_XLSX
    ) as writer:
        best_global_df.to_excel(
            writer,
            sheet_name="Best_Global",
            index=False,
        )

        best_by_k_df.to_excel(
            writer,
            sheet_name="Best_By_Global_K",
            index=False,
        )

        best_by_size_k_df.to_excel(
            writer,
            sheet_name="Best_By_Size_K",
            index=False,
        )

        per_machine_df.to_excel(
            writer,
            sheet_name="Per_Machine_Best_Global",
            index=False,
        )

        top_singles_df.to_excel(
            writer,
            sheet_name="Top_Global_Singles",
            index=False,
        )

        top_pairs_df.to_excel(
            writer,
            sheet_name="Top_Global_Pairs",
            index=False,
        )

        beam_df.to_excel(
            writer,
            sheet_name="Beam_History",
            index=False,
        )

        candidate_pool_df.to_excel(
            writer,
            sheet_name="Candidate_Pool",
            index=False,
        )

        config_df.to_excel(
            writer,
            sheet_name="Config",
            index=False,
        )

    # -------------------------------------------------------------------------
    # JSON.
    # -------------------------------------------------------------------------
    json_payload = {
        "description": (
            "Supervised development oracle search for ONE GLOBAL "
            "TDA feature subset and ONE GLOBAL k shared by all machines."
        ),
        "warning": (
            "Development test labels are used for global subset/k selection. "
            "This is an optimistic capability study, not an unseen-machine "
            "selection method."
        ),
        "pool_mode": POOL_MODE,
        "pool_source": pool_source,
        "candidate_pool_size": len(
            candidate_pool
        ),
        "candidate_pool": candidate_pool,
        "global_k_values": list(
            K_VALUES
        ),
        "primary_objective": (
            "HM across machine-level HM(AUC_source,AUC_target,pAUC@0.1)"
        ),
        "score_direction": (
            "higher mean kNN distance = anomaly; never inverted"
        ),
        "best_global": {
            "k": int(
                best_global[
                    "k"
                ]
            ),
            "n_features": int(
                best_global[
                    "n_features"
                ]
            ),
            "features": list(
                best_global[
                    "subset"
                ]
            ),
            "global_dcase_hm": float(
                best_global[
                    "global_dcase_hm"
                ]
            ),
            "global_component_hm": float(
                best_global[
                    "global_component_hm"
                ]
            ),
            "mean_machine_dcase_hm": float(
                best_global[
                    "mean_machine_dcase_hm"
                ]
            ),
            "min_machine_dcase_hm": float(
                best_global[
                    "min_machine_dcase_hm"
                ]
            ),
            "mean_auc_all": float(
                best_global[
                    "mean_auc_all"
                ]
            ),
            "mean_auc_source": float(
                best_global[
                    "mean_auc_source"
                ]
            ),
            "mean_auc_target": float(
                best_global[
                    "mean_auc_target"
                ]
            ),
            "mean_pauc_01": float(
                best_global[
                    "mean_pauc_01"
                ]
            ),
            "min_any_domain_auc": float(
                best_global[
                    "min_any_domain_auc"
                ]
            ),
            "per_machine": best_global[
                "machine_metrics"
            ],
        },
        "best_by_global_k": [
            {
                "k": int(
                    row[
                        "k"
                    ]
                ),
                "n_features": int(
                    row[
                        "n_features"
                    ]
                ),
                "features": list(
                    row[
                        "subset"
                    ]
                ),
                "global_dcase_hm": float(
                    row[
                        "global_dcase_hm"
                    ]
                ),
                "mean_machine_dcase_hm": float(
                    row[
                        "mean_machine_dcase_hm"
                    ]
                ),
                "min_machine_dcase_hm": float(
                    row[
                        "min_machine_dcase_hm"
                    ]
                ),
                "mean_auc_all": float(
                    row[
                        "mean_auc_all"
                    ]
                ),
            }
            for row in sorted(
                best_by_k_final,
                key=lambda r: r[
                    "k"
                ],
            )
        ],
    }

    with open(
        OUTPUT_JSON,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            json_payload,
            f,
            indent=2,
            ensure_ascii=False,
        )

    print(
        "\nSaved:"
    )
    print(
        f"  {OUTPUT_XLSX}"
    )
    print(
        f"  {OUTPUT_JSON}"
    )

    print(
        "\nEvaluated unique feature subsets:"
    )
    print(
        f"  {len(evaluator.cache):,}"
    )


if __name__ == "__main__":
    main()